# Load Image Data

In [ ]:
VERBOSE_LOGS = False

In [ ]:
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from skimage.color import rgb2gray

# Project and dataset paths
PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "data" / "KTH-TIPS2-b"

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

# Find all image files
image_files = sorted(path for path in DATASET_DIR.rglob("*") if path.is_file() and path.suffix.lower() == ".png")

if not image_files:
    raise RuntimeError("No images found in the dataset directory.")

# Organise images by material
images_by_material = defaultdict(list)
dataset_records = []

for image_path in image_files:
    # First subdirectory identifies the material, second subdirectory identifies the physical sample
    relative_path = image_path.relative_to(DATASET_DIR)
    material = relative_path.parts[0]
    sample = relative_path.parts[1] if len(relative_path.parts) > 2 else "unknown"

    images_by_material[material].append(image_path)
    dataset_records.append({
        "image_path": str(image_path),
        "material": material,
        "sample": sample,
        "filename": image_path.name
    })

# Convert dataset_records to dataframe, sort for consistency
df = pd.DataFrame(dataset_records)
df = df.sort_values(by=["material", "sample", "filename"]).reset_index(drop=True)

# Add labels
label_encoder = LabelEncoder()
df["material_label"] = label_encoder.fit_transform(df["material"])

# Create material_datasets variable
material_datasets = {}
for material in label_encoder.classes_:
    material_df = df[df["material"] == material]
    material_datasets[material] = {
        "paths": material_df["image_path"].tolist(),
        "samples": material_df["sample"].tolist(),
        "labels": material_df["material_label"].tolist()
    }

In [ ]:
def load_image(image_path, grayscale=False):
    """
    Load an image from disk.

    Parameters
    ----------
    image_path : str or Path
        Image location.
    grayscale : bool
        Whether to convert the image to grayscale.

    Returns
    -------
    numpy.ndarray
        Image array.
    """

    image = Image.open(image_path).convert("RGB")

    image = np.asarray(image)

    if grayscale:
        image = rgb2gray(image)

    return image

In [ ]:
# If verbose logging is enabled, show:
if VERBOSE_LOGS:

# DATABASE STATISTICS
    print(f"Total images discovered: {len(df)}")
    print(f"Number of materials: {df['material'].nunique()}")
    print(f"Number of samples: {df['sample'].nunique()}")

    material_counts = df.groupby("material").size()
    print("Images per material:")
    display(material_counts.to_frame(name="image_count"))

    sample_counts = pd.crosstab(df["material"], df["sample"])
    display(sample_counts)

    print("\nDataset summary:")
    print(f"Materials: {df['material'].nunique()}")
    print(f"Images: {len(df)}")
    print(f"Samples: {df.groupby('material')['sample'].nunique().sum()}")

# MATERIAL-TO-LABEL MAPPING
    material_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))

    print("Material label mapping:")
    for material, label in material_mapping.items():
        print(f"{label}: {material}")

# PRINT EXAMPLE IMAGE
    example = df.iloc[0]
    image = load_image(example["image_path"])

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f"Material: {example['material']}")
    plt.axis("off")
    plt.show()

print("Successfully loaded dataset.")